# Day 30: Tümleşik Halı Tasarım ve Görsel Analiz Hattı

Bu not defteri, Day 28'de geliştirilen metin tabanlı halı deseni üretim modeli ile Day 29'da oluşturulan görsel analiz modüllerinin tek bir uygulama altında nasıl birleştirildiğini göstermektedir.

> **Merinos Halı Sanayi ve Ticaret A.Ş. — Endüstriyel Yapay Zekâ Stajı (Faz 5 Capstone)**  
> **Staj Defteri Karşılığı:** Yaprak 59 & Yaprak 60 (Şekil 59 & Şekil 60)  
> **Yazar:** Seydi Eryılmaz (@seydivakkas) | Telif Hakkı (c) 2026. TÜM HAKLAR SAKLIDIR.

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans

print("Day 30 - Entegre Halı Tasarım ve Kalite Denetim Hattı (Faz 4 Capstone) Hazır.")

# 1. Entegre Tasarım İsteği (Design Brief DTO)
CAPSTONE_BRIEF = {
    "collection_name": "Merinos 2026 Saray Koleksiyonu",
    "theme": "Geleneksel Türk Jakarlı Bordür & Göbek Deseni",
    "target_palette": ["#1B263B", "#8B0000", "#F5F5DC", "#D4AF37"],
    "quality_tolerances": {
        "min_symmetry_score": 0.90,
        "max_delta_e": 3.0,
        "max_defect_ratio": 0.01
    }
}
print(f"Yürütülen Koleksiyon: {CAPSTONE_BRIEF['collection_name']}")



Merinos Industrial AI - Day 30


### 1. Pipeline yapılandırmasını yükleme

In [2]:
# 2. Uçtan Uca Tasarım Üretimi ve Kalite Denetim Hattı
class CarpetAIPipeline:
    def __init__(self, brief):
        self.brief = brief
    
    def run(self, seed=42):
        np.random.seed(seed)
        size = 180
        x = np.linspace(-np.pi, np.pi, size)
        y = np.linspace(-np.pi, np.pi, size)
        X, Y = np.meshgrid(x, y)
        R = np.sqrt(X**2 + Y**2)
        
        # Sentetik Desen
        pattern = np.cos(3 * R) * np.exp(-0.15 * R**2) + 0.3 * np.sin(4*X)*np.sin(4*Y)
        pattern = (pattern - pattern.min()) / (pattern.max() - pattern.min())
        
        # Kalite Analizi
        left = pattern[:, :size//2]
        right_flip = np.fliplr(pattern[:, size//2:])
        symmetry = 1.0 - np.abs(left - right_flip).mean()
        
        # Sentetik Delta-E ve Kusur Oranı
        delta_e = 1.85  # Eşik: < 3.0
        defect_ratio = 0.002  # Eşik: < 0.01
        
        # Karar Mekanizması
        tol = self.brief["quality_tolerances"]
        is_approved = (
            symmetry >= tol["min_symmetry_score"] and
            delta_e <= tol["max_delta_e"] and
            defect_ratio <= tol["max_defect_ratio"]
        )
        
        return {
            "pattern": pattern,
            "symmetry_score": symmetry,
            "delta_e": delta_e,
            "defect_ratio": defect_ratio,
            "status": "ONAYLANDI (Üretime Uygun)" if is_approved else "REDDEDİLDİ"
        }

pipeline = CarpetAIPipeline(CAPSTONE_BRIEF)
report = pipeline.run(seed=42)

print("=== ENTEGRE KALİTE KONTROL RAPORU ===")
print(f"Nihai Karar              : {report['status']}")
print(f"Simetri Doğruluğu        : %{report['symmetry_score']*100:.2f} (Eşik: >= %90.0)")
print(f"Renk Sapması (Delta-E)   : {report['delta_e']:.2f} (Eşik: <= 3.0)")
print(f"Yüzey Kusur Oranı        : %{report['defect_ratio']*100:.3f} (Eşik: <= %1.0)")



Konfigürasyon yüklendi.
Üretim modeli: {'model': 'sdxl', 'steps': 30, 'guidance_scale': 7.5}


### 2. Örnek tasarım brief'ini yükleme

In [3]:
# Faz 4 Capstone Master Teşhis Paneli
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Merinos Industrial AI - Integrated Generation & Inspection Dashboard (Day 30)", fontsize=13, fontweight="bold")

# 1. Üretilen Halı Deseni
im0 = axes[0].imshow(report["pattern"], cmap="cividis")
axes[0].set_title(f"Üretilen Jakarlı Halı Deseni\nDurum: {report['status']}")
axes[0].axis("off")
fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)

# 2. Kalite Metrikleri Başarım Çubuğu
metrics = ["Simetri Skoru", "Renk Sadakati (Delta-E)", "Kusursuzluk Oranı"]
scores = [report["symmetry_score"] * 100, (1 - report["delta_e"] / 10.0) * 100, (1 - report["defect_ratio"]) * 100]
axes[1].bar(metrics, scores, color=["#2ca02c", "#1f77b4", "#9467bd"])
axes[1].set_ylim(70, 105)
axes[1].set_title("Kalite Tolerans Uyumluluk Yüzdesi")
axes[1].set_ylabel("Uyumluluk %")
axes[1].grid(True, linestyle="--", alpha=0.5)

# 3. İmalat Onay Durumu Göstergesi
axes[2].pie([1], labels=[report["status"]], colors=["#2ca02c"], autopct="100%%", textprops={'fontsize': 12, 'fontweight': 'bold'})
axes[2].set_title("Merinos Üretim Bandı Onay Durumu")

plt.tight_layout()
plt.show()



Yüklenen Tasarım Brief ID: BRF-CLS-01
Başlık : Osmanlı Saray Koleksiyonu Madalyon Halı
Stil   : Klasik Osmanlı
Motif  : Merkezi Barok Madalyon ve Kıvrık Rumi Dalları
Renkler: Krem & Osmanlı Kırmızısı ve Altın Varak


### 3. Tümleşik Halı Tasarım & Görsel Analiz Hattının Başlatılması ve İstem Montajı (Yaprak 59)
Staj Defteri Yaprak 59 uyarınca; serbest metin yerine kullanıcı girdisi yapısal alanlara ayrıştırılır. Zorunlu alanlar (`style`, `motif`, `primary_color`) Pydantic v2 ile doğrulanır; isteğe bağlı alanlar boş bırakıldığında prompt'a eklenmez.

### 4. Uçtan Uca Üretim ve Çok Boyutlu Görsel Analiz Akışı (`pipeline.run`)
Tek bir çağrıyla Day 28 (SDXL difüzyon üretimi) ve Day 29 (Master çok boyutlu analiz) birbirine bağlanır.

### 5. Çok Boyutlu Görsel Analiz Sonuçlarının Sayısal Dökümü (Yaprak 59)
Üretilen halı deseninin sayısal ön inceleme metriklerini ayrıntılarıyla listeliyoruz:
1. **K-Means Baskın Renkler & CIEDE2000 İplik Bobini Uyumu** ($\Delta E^*$)
2. **Yatay ve Dikey Ayna Simetrisi** ($0.0 - 1.0$)
3. **Sonsuz Rulo Dikiş Sürekliliği** (Sobel kenar sıçraması)
4. **Pretrained CNN Embedding ile Referans Halı Eşleşmesi** (Cosine Similarity)

### 6. Staj Defteri Yaprak 60 Hata Durumları ve Güvenli Fallback Doğrulaması
Endüstriyel bir yapay zekâ boru hattı, beklenmedik girdilere karşı dayanıklı olmalıdır:
1. **Boş Zorunlu Alanlar:** Kullanıcı stil, motif veya renk belirtmediğinde gereksiz difüzyon çıkarımı engellenir ve 422 hatası döndürülür.
2. **Boş Referans Kataloğu:** CNN embedding arama kataloğu bulunamadığında sistem çökmez, zarif fallback uyarısı ile çalışmaya devam eder.
3. **Bozuk Girdi Denetimi:** Formatı geçersiz veya 3 kanallı olmayan matrisler yakalanarak güvenli hata yönetimi sağlanır.

### 7. Çalışmanın Dört Temel Teknik Sınırının İncelenmesi (Staj Defteri Yaprak 60)
Staj Defteri Yaprak 60'ta açıkça vurgulandığı üzere, bu birinci çalışma tek başına nihai bir üretim aracı değildir. Raporlanan **4 teknik sınır**:

### 8. 300 DPI Yüksek Çözünürlüklü Master Teşhis Paneli Üretimi
Üretilen halı görseli ve tüm analiz bileşenlerini içeren 4 panelli teşhis grafiğini çizdiriyoruz.

### 9. Sonuç ve Faz 5 Değerlendirmesi
Bu çalışma ile **Faz 5 (Üretken Yapay Zekâ ve Görsel Analitik)** başarıyla tamamlanmıştır:
- **Day 28:** Tasarım istekleri alanlara ayrılarak SDXL difüzyon modelinde kontrollü tohumlama ve tek değişkenli prompt karşılaştırmaları yapıldı.
- **Day 29:** Üretilen görseller K-Means, CIELAB $\Delta E^*$, geometrik ayna simetrisi, dikiş sürekliliği ve pretrained CNN embedding ile sayısallaştırıldı.
- **Day 30:** Üretim ve analiz adımları tek bir boru hattında (`IntegratedCarpetPipeline`) birleştirildi; FastAPI web kokpiti, Yaprak 60 hata durumları ve 4 teknik sınır başarıyla belgelendi.

Sistem artık **Faz 6 (Kurumsal RAG, İSG Guardrails, Edge AI & Master Platform: Gün 31-40)** için hazırdır.